# Support Vector Machine (SVM) Benchmarking

In this notebook, we compare our custom-built Linear SVM (optimized via Stochastic Gradient Descent and Hinge Loss) against the industry-standard `scikit-learn` implementation. We will evaluate the performance on the Breast Cancer dataset.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
from time import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC as SklearnSVC

# Import our custom modules
from classical_ml.svm.svm import SVM as CustomSVM
from utils.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
# 1. Load Dataset
data = load_breast_cancer()
X, y = data.data, data.target

# 2. Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Scale the features (Crucial for SVM Margin Calculation)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training data shape: {X_train_scaled.shape}")
print(f"Testing data shape: {X_test_scaled.shape}")

Training data shape: (455, 30)
Testing data shape: (114, 30)


In [3]:
print("1. Custom Linear SVM")
start = time()

# Initialize SVM with a small learning rate and lambda for regularization
custom_svm = CustomSVM(learning_rate=0.001, lambda_param=0.01, n_iters=1000)
custom_svm.fit(X_train_scaled, y_train)
preds_custom = custom_svm.predict(X_test_scaled)
time_custom = time() - start

print(f"Accuracy  : {accuracy_score(y_test, preds_custom):.4f}")
print(f"Precision : {precision_score(y_test, preds_custom):.4f}")
print(f"Recall    : {recall_score(y_test, preds_custom):.4f}")
print(f"F1-Score  : {f1_score(y_test, preds_custom):.4f}")
print(f"Time Taken: {time_custom:.5f} seconds\n")

1. Custom Linear SVM
Accuracy  : 0.9825
Precision : 0.9726
Recall    : 1.0000
F1-Score  : 0.9861
Time Taken: 3.40991 seconds



In [4]:
print("2. Scikit-Learn SVM (Linear Kernel)")
start = time()

# Using linear kernel to match our custom implementation
sk_svm = SklearnSVC(kernel='linear')
sk_svm.fit(X_train_scaled, y_train)
preds_sk = sk_svm.predict(X_test_scaled)
time_sk = time() - start

print(f"Accuracy  : {accuracy_score(y_test, preds_sk):.4f}")
print(f"Precision : {precision_score(y_test, preds_sk):.4f}")
print(f"Recall    : {recall_score(y_test, preds_sk):.4f}")
print(f"F1-Score  : {f1_score(y_test, preds_sk):.4f}")
print(f"Time Taken: {time_sk:.5f} seconds\n")

2. Scikit-Learn SVM (Linear Kernel)
Accuracy  : 0.9561
Precision : 0.9714
Recall    : 0.9577
F1-Score  : 0.9645
Time Taken: 0.04679 seconds



## Conclusion
Our custom Linear SVM performs exceptionally well, achieving an accuracy and F1-score highly comparable to Scikit-Learn's implementation. 

The main difference lies in the underlying optimization technique. Our custom model uses **Stochastic Gradient Descent (SGD)** to minimize the Hinge Loss function over 1000 iterations. On the other hand, `scikit-learn` uses highly optimized C/C++ libraries (`libsvm` or `liblinear`) that solve the dual optimization problem using techniques like Sequential Minimal Optimization (SMO), which can be computationally faster and more precise in defining the exact support vectors. However, our pure NumPy implementation proves that the fundamental mathematics of the margin and hinge loss hold true.